# GraphRouter - Training

This notebook demonstrates how to train the **GraphRouter** (Graph Neural Network Router).

## Overview

GraphRouter uses a Graph Neural Network (GNN) to model the relationships between queries and LLMs.
It constructs a heterogeneous graph where queries and LLMs are nodes, and performance scores are edge weights.

**Key Features**:
- Graph-based representation of query-LLM interactions
- Message passing for learning representations
- Can capture complex relational patterns

## 1. Environment Setup

In [1]:
# Install required packages (for Colab)
!git clone https://github.com/ulab-uiuc/LLMRouter.git
%cd LLMRouter
!pip install -e .
!pip install torch torch-geometric


Cloning into 'LLMRouter'...
remote: Enumerating objects: 5897, done.
remote: Counting objects: 100% (220/220), done.
remote: Compressing objects: 100% (136/136), done.
remote: Total 5897 (delta 109), reused 119 (delta 84), pack-reused 5677 (from 1)
Receiving objects: 100% (5897/5897), 88.90 MiB | 50.58 MiB/s, done.
Resolving deltas: 100% (2880/2880), done.
Updating files: 100% (280/280), done.
/home/zhongjie/LLMRouter
Obtaining file:///home/zhongjie/LLMRouter
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for llmrouter-lib (pyproject.toml) ... done
  Created wheel for llmrouter-lib: filename=llmrouter_lib-0.1.1-0.editable-py3-none-any.whl size=14451 sha256=56b69718b77669f1ae7af89e2f02ce48c162fc47ec91589e7d06a5d9039127d8
  Stored in directory: /tmp/pip-ephem-wheel-cache-dw036n5u/wheels/82/4a/fd/59c4aec93c356c

In [ ]:
import os
os.environ['OPENAI_API_KEY'] = 'your-key'
os.environ['ANTHROPIC_API_KEY'] = 'your-key'
# Or for multiple keys:
os.environ['API_KEYS'] = '["key1", "key2"]'

In [ ]:
import torch
from llmrouter.models.graphrouter import GraphRouter, GraphTrainer
from llmrouter.utils import setup_environment

setup_environment()

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## 2. Configuration

GraphRouter uses the following configuration parameters:

| Parameter | Description | Default |
|-----------|-------------|--------|
| `hidden_dim` | GNN hidden layer dimension | 64 |
| `learning_rate` | Learning rate | 0.001 |
| `weight_decay` | L2 regularization | 0.0001 |
| `train_epoch` | Training epochs | 100 |
| `batch_size` | Batch size | 4 |
| `train_mask_rate` | Edge masking rate | 0.3 |
| `val_split_ratio` | Validation split | 0.2 |

In [3]:
import yaml

CONFIG_PATH = "configs/model_config_train/graphrouter.yaml"

with open(CONFIG_PATH, 'r') as f:
    config = yaml.safe_load(f)

print("Current Configuration:")
print("=" * 50)
print(yaml.dump(config, default_flow_style=False))

Current Configuration:
data_path:
  llm_data: data/example_data/llm_candidates/default_llm.json
  llm_embedding_data: data/example_data/llm_candidates/default_llm_embeddings.json
  query_data_test: data/example_data/query_data/default_query_test.jsonl
  query_data_train: data/example_data/query_data/default_query_train.jsonl
  query_embedding_data: data/example_data/routing_data/query_embeddings_longformer.pt
  routing_data_test: data/example_data/routing_data/default_routing_test_data.jsonl
  routing_data_train: data/example_data/routing_data/default_routing_train_data.jsonl
hparam:
  batch_size: 4
  hidden_dim: 64
  learning_rate: 0.001
  random_state: 42
  train_epoch: 100
  train_mask_rate: 0.3
  val_split_ratio: 0.2
  weight_decay: 0.0001
metric:
  weights:
    cost: 0
    llm_judge: 0
    performance: 1
model_path:
  ini_model_path: ''
  load_model_path: saved_models/graphrouter/graphrouter.pt
  save_model_path: saved_models/graphrouter/graphrouter.pt



## 3. Initialize Router

In [4]:
router = GraphRouter(yaml_path=CONFIG_PATH)

print("Router initialized successfully!")
print(f"Number of training samples: {len(router.routing_data_train)}")
print(f"Number of LLM candidates: {len(router.llm_data)}")
print(f"LLM candidates: {list(router.llm_data.keys())}")

✅ MetaRouter initialized successfully (YAML + data loaded).
Router initialized successfully!
Number of training samples: 50544
Number of LLM candidates: 14
LLM candidates: ['qwen2.5-7b-instruct', 'codegemma-7b', 'gemma-2-9b-it', 'llama-3.1-8b-instruct', 'llama3-chatqa-1.5-8b', 'mistral-7b-instruct-v0.3', 'llama-3.3-nemotron-super-49b-v1', 'llama-3.1-nemotron-51b-instruct', 'llama3-chatqa-1.5-70b', 'llama3-70b-instruct', 'mixtral-8x7b-instruct-v0.1', 'mixtral-8x22b-instruct-v0.1', 'palmyra-creative-122b', 'mistral-nemo-12b-instruct']


## 4. Graph Structure Visualization

In [5]:
# Understand the graph structure
print("Graph Structure Information:")
print("=" * 50)
print(f"\nNode types:")
print(f"  - Query nodes: Based on training queries")
print(f"  - LLM nodes: {len(router.llm_data)} models")
print(f"\nEdge types:")
print(f"  - Query -> LLM edges (performance scores)")
print(f"\nThe GNN learns to predict missing edges for new queries.")

Graph Structure Information:

Node types:
  - Query nodes: Based on training queries
  - LLM nodes: 14 models

Edge types:
  - Query -> LLM edges (performance scores)

The GNN learns to predict missing edges for new queries.


## 5. Training

In [6]:
trainer = GraphTrainer(router=router, device=device)

print("Trainer initialized!")
print(f"Device: {device}")
print(f"Save path: {trainer.save_model_path}")

Trainer initialized!
Device: cuda
Save path: /home/zhongjie/LLMRouter/llmrouter/saved_models/graphrouter/graphrouter.pt


In [7]:
print("Starting training...")
print("=" * 50)

best_result = trainer.train()

print("=" * 50)
print("Training completed!")
if best_result:
    print(f"Best validation result: {best_result}")

Starting training...


/home/zhongjie/LLMRouter/llmrouter/models/graphrouter/graph_nn.py:136: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.train_mask = torch.tensor(data.train_mask, dtype=torch.bool)
/home/zhongjie/LLMRouter/llmrouter/models/graphrouter/graph_nn.py:137: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.valide_mask = torch.tensor(data.valide_mask, dtype=torch.bool)


Epoch 0: train_loss=1.2877, val_result=0.1774
Epoch 10: train_loss=0.4122, val_result=0.6208
Epoch 20: train_loss=0.4345, val_result=0.1774
Epoch 30: train_loss=0.3860, val_result=0.6208
Epoch 40: train_loss=0.3645, val_result=0.1800
Epoch 50: train_loss=0.3542, val_result=0.1782
Epoch 60: train_loss=0.3482, val_result=0.3299
Epoch 70: train_loss=0.3493, val_result=0.1827
Epoch 80: train_loss=0.3452, val_result=0.2612
Epoch 90: train_loss=0.3459, val_result=0.2193
Training completed. Best validation result: 0.6208
Training completed!
Best validation result: 0.6208158731460571


## 6. Model Verification

In [9]:
# Verify the trained model
import os
model_path = trainer.save_model_path
if os.path.exists(model_path):
    checkpoint = torch.load(model_path, map_location='cpu')
    print(f"Model loaded from: {model_path}")
else:
    print(f"Model not found at: {model_path}")

Model loaded from: /home/zhongjie/LLMRouter/llmrouter/saved_models/graphrouter/graphrouter.pt


In [10]:
# Test prediction
test_query = {"query": "What is the capital of France?"}
result = router.route_single(test_query)

print(f"Test query: {test_query['query']}")
print(f"Routed to: {result['model_name']}")

Input ids are automatically padded to be a multiple of `config.attention_window`: 512


Test query: What is the capital of France?
Routed to: llama3-chatqa-1.5-8b


## Summary

In this notebook, we:

1. **Loaded Configuration**: Set up GraphRouter with YAML configuration
2. **Understood Graph Structure**: Query-LLM bipartite graph
3. **Trained GNN Model**: Used message passing to learn representations
4. **Verified Model**: Tested routing with sample queries

**Key Takeaways**:
- GraphRouter models query-LLM relationships as a graph
- GNN can capture complex interaction patterns
- Edge masking during training improves generalization

**Next Steps**:
- Use the next part of notebook for inference
- Experiment with different GNN architectures

# GraphRouter - Inference

This part of notebook demonstrates how to use a trained **GraphRouter** for inference.

## 1. Environment Setup

## 2. Load Trained Router

In [11]:
CONFIG_PATH = "configs/model_config_test/graphrouter.yaml"

with open(CONFIG_PATH, 'r') as f:
    config = yaml.safe_load(f)

router = GraphRouter(yaml_path=CONFIG_PATH)
print(f"Router loaded with {len(router.llm_data)} LLM candidates")

✅ MetaRouter initialized successfully (YAML + data loaded).
Router loaded with 14 LLM candidates


## 3. Query Routing

In [12]:
EXAMPLE_QUERIES = [
    {"query": "What is the capital of France?"},
    {"query": "Solve the equation: 2x + 5 = 15"},
    {"query": "Write a Python function to check if a number is prime."},
    {"query": "Explain quantum computing in simple terms."},
]

print("Routing Results:")
print("=" * 60)

for i, query in enumerate(EXAMPLE_QUERIES, 1):
    result = router.route_single(query)
    print(f"{i}. {query['query'][:50]}...")
    print(f"   Routed to: {result['model_name']}")

Routing Results:
1. What is the capital of France?...
   Routed to: llama3-chatqa-1.5-8b
2. Solve the equation: 2x + 5 = 15...
   Routed to: llama-3.1-nemotron-51b-instruct
3. Write a Python function to check if a number is pr...
   Routed to: llama-3.1-nemotron-51b-instruct
4. Explain quantum computing in simple terms....
   Routed to: llama-3.1-nemotron-51b-instruct


## 4. File-Based Inference

Load queries from a file and save results.

In [15]:
import json

# Load queries from a JSONL file
def load_queries_from_file(file_path):
    """Load queries from a JSONL file."""
    queries = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                queries.append(json.loads(line))
    return queries

# Save results to a JSONL file
def save_results_to_file(results, output_path):
    """Save routing results to a JSONL file."""
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, 'w', encoding='utf-8') as f:
        for result in results:
            f.write(json.dumps(result, ensure_ascii=False) + '\n')
    print(f"Results saved to: {output_path}")

# Example: Load from default query file
QUERY_FILE = "data/example_data/query_data/default_query_test.jsonl"
OUTPUT_FILE = "outputs/graphrouter_results.jsonl"

if os.path.exists(QUERY_FILE):
    # Load queries
    file_queries = load_queries_from_file(QUERY_FILE)
    print(f"Loaded {len(file_queries)} queries from: {QUERY_FILE}")
    
    # Route queries
    file_results = router.route_batch(batch=file_queries[:10])
    print(f"Routed {len(file_results)} queries")
    
    # Save results
    save_results_to_file(file_results, OUTPUT_FILE)
    
    # Show sample results
    print(f"\nSample results:")
    for i, result in enumerate(file_results[:3], 1):
        print(f"  {i}. {result.get('query', '')[:40]}... -> {result['model_name']}")
else:
    print(f"Query file not found: {QUERY_FILE}")
    print("Create a JSONL file with format: {\"query\": \"Your question\"}")

Loaded 706 queries from: data/example_data/query_data/default_query_test.jsonl

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Gi

## Summary

This notebook demonstrated:
1. Loading a trained GraphRouter
2. Routing queries using GNN-based inference

GraphRouter is effective for:
- Capturing relational patterns between queries and LLMs
- Leveraging graph structure for better routing